# Simple RAG 2 (Multiple Documents) - Document Chunking and Embedding

**Flow:**

Documents → Chunk → Embed → Store (ChromaDB)

## Install Dependencies

In [1]:
# Conda environment setup
#!pip install langchain langchain-chroma langchain-openai chromadb pypdf

## Basic RAG Code Phase 1 - Document Ingestion and Embedding

In [2]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [3]:
# # Colab File Location Setup
# file_location = "/content/drive/MyDrive/rag_langchain/data/rag_sample1.txt"
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [4]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [5]:
# Local File Location Setup
data_folder = "../data"
store_location = "../vector_db/chroma_db2"

In [6]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()


True

### Experiment Config (Set Before Step 1)

Tune these values before running Step 1 so you can quickly test chunking behavior.


In [7]:
# Recommended defaults for this notebook's sample docs
CHUNK_SIZE = 500
CHUNK_OVERLAP = 250

print("Experiment config:")
print(f"- CHUNK_SIZE: {CHUNK_SIZE}")
print(f"- CHUNK_OVERLAP: {CHUNK_OVERLAP}")


Experiment config:
- CHUNK_SIZE: 500
- CHUNK_OVERLAP: 250


### Step 1: Load & split documents

In [8]:
from pathlib import Path
import re

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_base_metadata(file_path: Path, doc_type: str) -> dict:
    stem = file_path.stem.lower()
    match = re.search(r"(sn-\d+)", stem)
    episode_id = match.group(1) if match else None
    episode_number = int(episode_id.split("-")[1]) if episode_id else None

    metadata = {
        "source": str(file_path),
        "file_name": file_path.name,
        "doc_type": doc_type,
    }
    if episode_id is not None:
        metadata["episode_id"] = episode_id
    if episode_number is not None:
        metadata["episode_number"] = episode_number
    return metadata

def load_documents(data_dir: str) -> list[Document]:
    docs = []
    for file_path in sorted(Path(data_dir).rglob("*")):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()
        if suffix == ".txt":
            text = file_path.read_text(encoding="utf-8")
            metadata = build_base_metadata(file_path, doc_type="transcript")
            docs.append(
                Document(
                    page_content=text,
                    metadata=metadata
                )
            )
        elif suffix == ".pdf":
            base_metadata = build_base_metadata(file_path, doc_type="show_notes")
            reader = PdfReader(str(file_path))
            for page_number, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                if text:
                    metadata = {**base_metadata, "page": page_number}
                    docs.append(
                        Document(
                            page_content=text,
                            metadata=metadata
                        )
                    )

    return docs

docs = load_documents(data_folder)
if not docs:
    raise ValueError(f"No supported documents found in {data_folder}")

print(f"Loaded {len(docs)} document/page item(s)")
known_episodes = sorted({d.metadata.get("episode_id") for d in docs if d.metadata.get("episode_id")})
print(f"Detected {len(known_episodes)} episode(s)")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = splitter.split_documents(docs)
print(f"Created {len(splits)} chunks")

Loaded 520 document/page item(s)
Detected 13 episode(s)
Created 10251 chunks


#### Why this Step 1 pattern is effective for multiple documents

This code uses a single `load_documents()` function to standardize ingestion for different file types (`.txt` and `.pdf`) into one common `Document` format.

It is a strong design pattern for multi-document RAG because:
- It scales cleanly: `Path(...).rglob("*")` scans the whole folder tree, so adding files does not require code changes.
- It is type-aware: each file type gets the right parser, while unsupported files are safely ignored.
- It preserves traceability: metadata (`episode_id`, `episode_number`, `doc_type`, `file_name`, `source`, and `page` for PDFs) makes later retrieval answers auditable.
- It keeps responsibilities separated: loading/parsing is isolated from chunking, making Step 2+ easier to maintain and test.
- It improves retrieval quality: `RecursiveCharacterTextSplitter` creates overlapping chunks, which preserves context across boundaries and usually gives better semantic search results.


### Step 2: Create embeddings + store in Chroma

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=store_location
)